In [1]:
from numba import njit

In [2]:
import sys
sys.path.insert(0, r'C:\projects\hedge_fund')

import numpy as np
import pandas as pd
from scipy.optimize import differential_evolution
from data.store.duckdb_client import DuckDBClient
from data.store.s3_client import S3Client
import time

# ── Load data ──────────────────────────────────────────────────────
db = DuckDBClient()
s  = S3Client()

all_files  = s.list_files('processed/1s/USDINR26SEPFUT/')
all_files += s.list_files('processed/1s/USDINR26OCTFUT/')

all_df = []
for f in all_files:
    parts = f.split('/')
    sym   = parts[2]
    date  = parts[3].replace('.parquet','')
    try:
        df = db.read_parquet(f'processed/1s/{sym}/{date}.parquet')
        df['ist_sec'] = (df['ts_sec'] + 19800) % 86400
        df = df[(df['ist_sec'] >= 9*3600) &
                (df['ist_sec'] <= 17*3600)].reset_index(drop=True)
        df['date'] = date
        all_df.append(df)
    except:
        continue
db.close()

df = pd.concat(all_df, ignore_index=True).reset_index(drop=True)
print(f'Loaded: {len(df):,} bars')

# ── Signals per day ────────────────────────────────────────────────
TICK=0.0025; W_sig=30; NW=300
signal_dfs = []
for d in sorted(df['date'].unique()):
    sub = df[df['date']==d].copy()
    sub['bid_chg']    = sub['total_bid_qty'].diff()
    sub['ask_chg']    = sub['total_ask_qty'].diff()
    pu                = sub['close'].diff().abs() < TICK*0.5
    sub['cancel_imb'] = 0.0
    sub.loc[pu&(sub['bid_chg']<0),'cancel_imb'] -= sub['bid_chg'].abs()
    sub.loc[pu&(sub['ask_chg']<0),'cancel_imb'] += sub['ask_chg'].abs()
    sub['s_cancel']   = -sub['cancel_imb'].rolling(W_sig).mean()
    sub['s_imb']      =  sub['imbalance_last'].rolling(W_sig).mean()
    sub['mid']        = (sub['bid_p1']+sub['ask_p1'])/2
    sub['s_wmid']     = (sub['weighted_mid']-sub['mid']).rolling(10).mean()
    sub['s_momentum'] = -sub['close'].pct_change(30)
    for col in ['s_cancel','s_imb','s_wmid','s_momentum']:
        rs=sub[col].rolling(NW).std()+1e-9
        sub[col]=(sub[col]/rs).clip(-1,1)
    sub['score']     = (0.393*sub['s_cancel'] - 0.218*sub['s_imb']
                      - 0.320*sub['s_wmid']   - 0.069*sub['s_momentum'])
    sub['fwd_1800s'] = sub['close'].shift(-1800) - sub['close']
    sub['spread']    = sub['ask_p1'] - sub['bid_p1']
    signal_dfs.append(sub)

df_sig     = pd.concat(signal_dfs, ignore_index=True).reset_index(drop=True)
score_arr  = df_sig['score'].values.astype(np.float64)
fwd_arr    = df_sig['fwd_1800s'].values.astype(np.float64)
close_arr  = df_sig['close'].values.astype(np.float64)
spread_arr = df_sig['spread'].values.astype(np.float64)
tick_arr   = df_sig['tick_count'].values.astype(np.float64)
print(f'Signal df: {len(df_sig):,} bars')

# ── Pre-compute rolling features ONCE (not inside fitness) ─────────
def precompute_features(close, spread, tick, w):
    n           = len(close)
    price_range = np.zeros(n)
    vol_ratio   = np.zeros(n)
    spread_z    = np.zeros(n)
    for i in range(w, n):
        wc             = close[i-w:i]
        price_range[i] = wc.max() - wc.min()
        tc_m           = tick[i-w:i].mean()
        vol_ratio[i]   = tick[i] / (tc_m + 1e-9)
        sw             = spread[i-w:i]
        spread_z[i]    = (spread[i] - sw.mean()) / (sw.std() + 1e-9)
    return price_range, vol_ratio, spread_z

# Pre-compute for fixed window = 900s (15 min)
W_fixed = 900
print(f'Pre-computing features (w={W_fixed}s)...')
pr, vr, sz = precompute_features(close_arr, spread_arr, tick_arr, W_fixed)
print('Done.')

# ── Fast fitness (no rolling inside) ──────────────────────────────
def fitness_fast(params):
    thr_trend  = params[0]
    thr_dead   = params[1]
    thr_vol    = params[2]
    thr_spread = params[3]

    trending = (pr > thr_trend) | (sz > thr_spread)
    dead     = (pr < thr_dead) & (vr < thr_vol)
    mr       = ~trending & ~dead

    valid    = ~np.isnan(score_arr) & ~np.isnan(fwd_arr)
    mr_mask  = mr & valid
    trd_mask = trending & valid

    if mr_mask.sum() < 1000 or trd_mask.sum() < 100:
        return 0.0

    ic_mr  = np.corrcoef(score_arr[mr_mask],  fwd_arr[mr_mask])[0,1]
    ic_trd = np.corrcoef(score_arr[trd_mask], fwd_arr[trd_mask])[0,1]

    if np.isnan(ic_mr) or np.isnan(ic_trd):
        return 0.0

    return -(ic_mr - ic_trd) * mr_mask.mean()

# ── Run GA ─────────────────────────────────────────────────────────
bounds = [
    (0.02, 0.20),   # trend price range
    (0.005, 0.03),  # dead price range
    (0.20, 0.80),   # vol ratio dead
    (1.0,  3.0),    # spread z trend
]

print('Running GA...')
t0 = time.time()
result = differential_evolution(
    fitness_fast, bounds,
    seed=42, maxiter=50, popsize=15,
    tol=1e-4, workers=1, disp=True
)
print(f'\nDone in {time.time()-t0:.1f}s')
print(f'Best IC separation x MR%: {-result.fun:.4f}')
print(f'Trend thr: {result.x[0]:.4f} paise')
print(f'Dead thr:  {result.x[1]:.4f} paise')
print(f'Vol ratio: {result.x[2]:.4f}')
print(f'Spread z:  {result.x[3]:.4f}')

# Validate
trending_opt = (pr > result.x[0]) | (sz > result.x[3])
dead_opt     = (pr < result.x[1]) & (vr < result.x[2])
mr_opt       = ~trending_opt & ~dead_opt
valid        = ~np.isnan(score_arr) & ~np.isnan(fwd_arr)

print('\n=== Validation ===')
for mask, name in [(mr_opt&valid,'MR'),(trending_opt&valid,'TRENDING'),(dead_opt&valid,'DEAD')]:
    if mask.sum() < 10: continue
    ic  = np.corrcoef(score_arr[mask], fwd_arr[mask])[0,1]
    pct = mask.mean()*100
    print(f'  {name:<12} IC={ic:.4f}  pct={pct:.1f}%  n={mask.sum()}')

Loaded: 69,075 bars
Signal df: 69,075 bars
Pre-computing features (w=900s)...
Done.
Running GA...
differential_evolution step 1: f(x)= -0.05799661848024692
differential_evolution step 2: f(x)= -0.058998949998819286
differential_evolution step 3: f(x)= -0.058998949998819286
differential_evolution step 4: f(x)= -0.060031286020311374
differential_evolution step 5: f(x)= -0.060031286020311374
differential_evolution step 6: f(x)= -0.060031286020311374
differential_evolution step 7: f(x)= -0.060031286020311374
differential_evolution step 8: f(x)= -0.06020759486463818
differential_evolution step 9: f(x)= -0.06020759486463818
differential_evolution step 10: f(x)= -0.06020759486463818
differential_evolution step 11: f(x)= -0.06020759486463818
differential_evolution step 12: f(x)= -0.06020759486463818
differential_evolution step 13: f(x)= -0.06020759486463818
differential_evolution step 14: f(x)= -0.06020759486463818
differential_evolution step 15: f(x)= -0.06020759486463818
differential_evoluti

In [3]:
# Backtest with GA-optimized regime filter
df_sig['price_range'] = pd.Series(pr)
df_sig['vol_ratio']   = pd.Series(vr)
df_sig['spread_z']    = pd.Series(sz)
df_sig['trending']    = (pd.Series(pr) > result.x[0]) | (pd.Series(sz) > result.x[3])
df_sig['dead']        = (pd.Series(pr) < result.x[1]) & (pd.Series(vr) < result.x[2])
df_sig['regime']      = np.where(df_sig['trending'], 'TRENDING',
                         np.where(df_sig['dead'], 'DEAD', 'MR'))
df_sig['hour']        = df_sig['ist_sec'] // 3600

LOT_SIZE    = 1000
LOTS        = 20
THRESHOLD   = -0.6
HOLD_MAX    = 1800
TRADE_HOURS = [13, 14, 15]
EOD_IST     = 15*3600 + 55*60

all_trades = []

for d in sorted(df_sig['date'].unique()):
    day  = df_sig[df_sig['date']==d].copy().reset_index(drop=True)
    inv  = 0; ep = 0.0; et = 0

    for i, row in day.iterrows():
        if pd.isna(row['score']): continue
        ist    = int(row['ist_sec'])
        hour   = int(row['hour'])
        score  = float(row['score'])
        bid_p1 = float(row['bid_p1'])
        ask_p1 = float(row['ask_p1'])
        regime = row['regime']

        if ist >= EOD_IST:
            if inv == -1:
                pnl = (ep - ask_p1) * LOTS * LOT_SIZE
                all_trades.append({'date':d,'hour':hour,'pnl':pnl,
                                   'reason':'EOD','hold':ist-et,'regime':regime})
                inv = 0
            break

        if inv == -1 and (ist - et) >= HOLD_MAX:
            pnl = (ep - ask_p1) * LOTS * LOT_SIZE
            all_trades.append({'date':d,'hour':hour,'pnl':pnl,
                               'reason':'TIME','hold':ist-et,'regime':regime})
            inv = 0

        # Entry — only in MR regime
        if inv == 0 and hour in TRADE_HOURS and regime == 'MR':
            if score < THRESHOLD:
                ep = ask_p1; et = ist; inv = -1

if all_trades:
    df_t = pd.DataFrame(all_trades)
    pnls = df_t['pnl']
    wins = pnls[pnls > 0]
    print(f'=== Backtest with Regime Filter ===')
    print(f'Total trades: {len(df_t)}')
    print(f'Win rate:     {len(wins)/len(pnls)*100:.1f}%')
    print(f'Total PnL:    Rs.{pnls.sum():,.0f}')
    print(f'Avg PnL:      Rs.{pnls.mean():,.0f}')
    print(f'Max win:      Rs.{pnls.max():,.0f}')
    print(f'Max loss:     Rs.{pnls.min():,.0f}')
    print()
    print('Per day:')
    for d, g in df_t.groupby('date'):
        wr = (g['pnl']>0).mean()*100
        print(f'  {d}: {len(g):>3} trades  win={wr:.0f}%  Rs.{g["pnl"].sum():>8,.0f}')

=== Backtest with Regime Filter ===
Total trades: 19
Win rate:     42.1%
Total PnL:    Rs.5,450
Avg PnL:      Rs.287
Max win:      Rs.2,650
Max loss:     Rs.-650

Per day:
  2026-09-22:   3 trades  win=33%  Rs.   2,300
  2026-09-23:   3 trades  win=33%  Rs.    -550
  2026-09-24:   4 trades  win=0%  Rs.  -1,200
  2026-09-25:   5 trades  win=60%  Rs.   1,850
  2026-09-30:   4 trades  win=75%  Rs.   3,050
